# Ribo-Seq Scoring of Triple-Frame Regions

**Objective**: Score triple-frame regions with Ribo-Seq coverage from bigWig files, including per-frame counts.

**IMPORTANT**: This notebook requires:
1. Notebook 2 to be run first (needs TSV files with CDS context)
2. Forward and reverse strand bigWig files (user must provide paths)

**Key challenge**: Mapping transcript coordinates to genomic coordinates while maintaining frame information.

**Outputs**:
- Total Ribo-Seq counts per region
- Per-frame Ribo-Seq counts (frame 0, 1, 2)
- Ribo-Seq density (counts/codon)
- Frame balance metrics

**Runtime**: ~5-10 minutes (depends on bigWig file access speed)

In [ ]:
!pip install pyBigWig


In [ ]:
import pandas as pd
import numpy as np
import pyBigWig
import matplotlib.pyplot as plt
from collections import defaultdict
from pathlib import Path
import pybedtools

print("Libraries loaded successfully")
print(f"pyBigWig version: {pyBigWig.__version__ if hasattr(pyBigWig, '__version__') else 'installed'}")

## 1. Configuration: BigWig File Paths

**IMPORTANT**: Update these paths to point to your Ribo-Seq bigWig files!

In [ ]:
# USER: UPDATE THESE PATHS TO YOUR BIGWIG FILES IF THEY ARE NOT IN THE REPOSITORY ROOT
BASE_DIR = Path("..").resolve()
FORWARD_BIGWIG = str(BASE_DIR / "TransCODE_GENCODE_v45_RiboSeqORFs_RiboCrypt_all_forward_unique.bigWig")  # For + strand transcripts
REVERSE_BIGWIG = str(BASE_DIR / "TransCODE_GENCODE_v45_RiboSeqORFs_RiboCrypt_all_reverse_unique.bigWig")  # For - strand transcripts

# Verify files exist
fw_path = Path(FORWARD_BIGWIG)
rv_path = Path(REVERSE_BIGWIG)

if not fw_path.exists():
    print(f"⚠️  WARNING: Forward bigWig not found: {FORWARD_BIGWIG}")
    print("    Please update FORWARD_BIGWIG path above!")
else:
    print(f"✓ Forward bigWig found: {FORWARD_BIGWIG}")

if not rv_path.exists():
    print(f"⚠️  WARNING: Reverse bigWig not found: {REVERSE_BIGWIG}")
    print("    Please update REVERSE_BIGWIG path above!")
else:
    print(f"✓ Reverse bigWig found: {REVERSE_BIGWIG}")

## 2. Load Data from Notebook 2

In [ ]:
# Load results from Notebook 2 (with CDS context)
results_dir = Path('../results/triple_frame_cds_context')

print("Loading results from Notebook 2...")

df_stop_free = pd.read_csv(results_dir / 'stop_free_with_cds_context.tsv', sep='\t')
print(f"  Loaded {len(df_stop_free):,} stop-free regions")

df_atg_stop = pd.read_csv(results_dir / 'atg_stop_with_cds_context.tsv', sep='\t')
print(f"  Loaded {len(df_atg_stop):,} ATG-to-STOP regions")

print("\nAll data loaded successfully!")

In [ ]:
def load_transcript_exons(gtf_path):
    """Load exon coordinates for each transcript from GTF"""
    print(f"Loading transcript exons from {gtf_path}...")
    print("This may take a few minutes...")
    
    gtf = pybedtools.BedTool(gtf_path)
    
    transcript_data = {}
    
    for feature in gtf:
        if feature.fields[2] != 'exon':
            continue
        
        attrs = {}
        for attr in feature.fields[8].split(';'):
            attr = attr.strip()
            if '"' in attr:
                key, value = attr.split(' "', 1)
                attrs[key] = value.rstrip('"')
        
        transcript_id = attrs.get('transcript_id')
        if not transcript_id:
            continue
        
        tx_id_base = transcript_id.split('.')[0]
        
        if tx_id_base not in transcript_data:
            transcript_data[tx_id_base] = {
                'chrom': feature.chrom,
                'strand': feature.strand,
                'exons': []
            }
        
        transcript_data[tx_id_base]['exons'].append({
            'start': feature.start,
            'end': feature.end
        })
    
    # Sort exons by start position
    for tx_id in transcript_data:
        transcript_data[tx_id]['exons'] = sorted(
            transcript_data[tx_id]['exons'], 
            key=lambda x: x['start']
        )
    
    print(f"  Loaded exons for {len(transcript_data):,} transcripts")
    return transcript_data

transcript_data = load_transcript_exons('../annotation.gtf')

## 4. Core Functions for Coordinate Mapping and Scoring

In [ ]:
def map_transcript_to_genomic(tx_start, tx_end, exons, strand):
    """
    Map transcript coordinates to genomic intervals.
    Handles splicing - may return multiple genomic intervals.
    
    Returns: list of (genomic_start, genomic_end, tx_offset_start) tuples
    tx_offset_start is the transcript position at the start of this genomic interval
    """
    genomic_intervals = []
    
    if strand == '+':
        # Forward strand: exons are in order
        tx_pos = 0
        
        for exon in exons:
            exon_length = exon['end'] - exon['start']
            exon_tx_start = tx_pos
            exon_tx_end = tx_pos + exon_length
            
            # Check if this exon overlaps our region
            if exon_tx_end > tx_start and exon_tx_start < tx_end:
                # Calculate overlap
                region_start_in_exon = max(0, tx_start - exon_tx_start)
                region_end_in_exon = min(exon_length, tx_end - exon_tx_start)
                
                genomic_start = exon['start'] + region_start_in_exon
                genomic_end = exon['start'] + region_end_in_exon
                tx_offset = exon_tx_start + region_start_in_exon
                
                genomic_intervals.append((genomic_start, genomic_end, tx_offset))
            
            tx_pos += exon_length
    
    else:  # Reverse strand
        # Reverse strand: exons are in reverse order for transcript coordinates
        tx_pos = 0
        
        for exon in reversed(exons):
            exon_length = exon['end'] - exon['start']
            exon_tx_start = tx_pos
            exon_tx_end = tx_pos + exon_length
            
            if exon_tx_end > tx_start and exon_tx_start < tx_end:
                region_start_in_exon = max(0, tx_start - exon_tx_start)
                region_end_in_exon = min(exon_length, tx_end - exon_tx_start)
                
                # For reverse strand, positions are reversed within exon
                genomic_end = exon['end'] - region_start_in_exon
                genomic_start = exon['end'] - region_end_in_exon
                tx_offset = exon_tx_start + region_start_in_exon
                
                genomic_intervals.append((genomic_start, genomic_end, tx_offset))
            
            tx_pos += exon_length
    
    return genomic_intervals

print("Coordinate mapping function defined")

In [ ]:
def score_region_with_riboseq(row, transcript_data, fw_bw, rv_bw):
    """
    Score a triple-frame region with Ribo-Seq counts from bigWig files.
    
    Returns dict with:
    - total_counts: total Ribo-Seq coverage
    - frame0_counts, frame1_counts, frame2_counts: per-frame counts
    - riboseq_density: counts per codon
    """
    # Extract transcript ID (remove version if present)
    tx_id_full = row['transcript_id']
    tx_id = tx_id_full.split('|')[0].split('.')[0] if '|' in tx_id_full else tx_id_full.split('.')[0]
    
    # Get transcript data
    if tx_id not in transcript_data:
        return {
            'total_counts': 0,
            'frame0_counts': 0,
            'frame1_counts': 0,
            'frame2_counts': 0,
            'riboseq_density': 0.0,
            'error': 'transcript_not_found'
        }
    
    tx_data = transcript_data[tx_id]
    chrom = tx_data['chrom']
    # Fix chromosome naming: bigWig uses '1' but GTF uses 'chr1'
    chrom = chrom.replace('chr', '') if chrom.startswith('chr') else chrom
    strand = tx_data['strand']
    exons = tx_data['exons']
    
    # Select appropriate bigWig based on strand
    bw = fw_bw if strand == '+' else rv_bw
    
    # Map transcript region to genomic intervals
    tx_start = int(row['region_start'])
    tx_end = int(row['region_end'])
    
    genomic_intervals = map_transcript_to_genomic(tx_start, tx_end, exons, strand)
    
    if not genomic_intervals:
        return {
            'total_counts': 0,
            'frame0_counts': 0,
            'frame1_counts': 0,
            'frame2_counts': 0,
            'riboseq_density': 0.0,
            'error': 'no_genomic_mapping'
        }
    
    # Initialize per-frame counters
    frame_counts = {0: 0.0, 1: 0.0, 2: 0.0}
    total_counts = 0.0
    
    # Extract counts from bigWig for each genomic interval
    try:
        for g_start, g_end, tx_offset_start in genomic_intervals:
            # Get per-nucleotide counts from bigWig
            try:
                counts = bw.values(chrom, g_start, g_end, numpy=True)
                if counts is None:
                    counts = np.zeros(g_end - g_start)
                else:
                    # Replace NaN with 0
                    counts = np.nan_to_num(counts, 0.0)
            except:
                counts = np.zeros(g_end - g_start)
            
            # Assign each genomic position to its transcript-frame position.
            # On reverse-strand transcripts, bigWig values are returned in
            # increasing genomic order, which is decreasing transcript order.
            interval_length = g_end - g_start
            for i, count in enumerate(counts):
                if strand == '-':
                    tx_pos = tx_offset_start + (interval_length - 1 - i)
                else:
                    tx_pos = tx_offset_start + i
                frame = tx_pos % 3
                frame_counts[frame] += count
                total_counts += count
    
    except Exception as e:
        return {
            'total_counts': 0,
            'frame0_counts': 0,
            'frame1_counts': 0,
            'frame2_counts': 0,
            'riboseq_density': 0.0,
            'error': str(e)
        }
    
    # Calculate density (counts per codon)
    region_length_codons = row['region_length_codons']
    density = total_counts / region_length_codons if region_length_codons > 0 else 0.0
    
    return {
        'total_counts': total_counts,
        'frame0_counts': frame_counts[0],
        'frame1_counts': frame_counts[1],
        'frame2_counts': frame_counts[2],
        'riboseq_density': density,
        'error': None
    }

print("Ribo-Seq scoring function defined")


## 5. Open BigWig Files

In [ ]:
print("Opening bigWig files...")

try:
    fw_bw = pyBigWig.open(FORWARD_BIGWIG)
    print(f"  ✓ Opened forward bigWig: {FORWARD_BIGWIG}")
    print(f"    Chromosomes: {len(fw_bw.chroms())}")
except Exception as e:
    print(f"  ✗ Error opening forward bigWig: {e}")
    fw_bw = None

try:
    rv_bw = pyBigWig.open(REVERSE_BIGWIG)
    print(f"  ✓ Opened reverse bigWig: {REVERSE_BIGWIG}")
    print(f"    Chromosomes: {len(rv_bw.chroms())}")
except Exception as e:
    print(f"  ✗ Error opening reverse bigWig: {e}")
    rv_bw = None

if fw_bw is None or rv_bw is None:
    print("\n⚠️  WARNING: Could not open bigWig files!")
    print("Please check the file paths in the configuration cell above.")
else:
    print("\n✓ Both bigWig files opened successfully!")

## 6. Score All Regions (Stop-Free Analysis)

In [ ]:
if fw_bw and rv_bw:
    print("\n" + "="*70)
    print("SCORING STOP-FREE REGIONS WITH RIBO-SEQ")
    print("="*70)
    
    results = []
    errors = 0
    
    for idx, row in df_stop_free.iterrows():
        if idx % 10000 == 0:
            print(f"  Processed {idx:,} / {len(df_stop_free):,} regions...")
        
        score = score_region_with_riboseq(row, transcript_data, fw_bw, rv_bw)
        results.append(score)
        
        if score.get('error'):
            errors += 1
    
    # Add scores to dataframe
    df_stop_free['total_riboseq_counts'] = [r['total_counts'] for r in results]
    df_stop_free['frame0_riboseq_counts'] = [r['frame0_counts'] for r in results]
    df_stop_free['frame1_riboseq_counts'] = [r['frame1_counts'] for r in results]
    df_stop_free['frame2_riboseq_counts'] = [r['frame2_counts'] for r in results]
    df_stop_free['riboseq_density'] = [r['riboseq_density'] for r in results]
    
    print(f"\n✓ Scored {len(df_stop_free):,} regions")
    print(f"  Errors: {errors:,} regions")
    print(f"  Regions with counts > 0: {(df_stop_free['total_riboseq_counts'] > 0).sum():,}")
else:
    print("⚠️  Skipping scoring - bigWig files not available")

## 7. Score All Regions (ATG-to-STOP Analysis)

In [ ]:
if fw_bw and rv_bw:
    print("\n" + "="*70)
    print("SCORING ATG-TO-STOP REGIONS WITH RIBO-SEQ")
    print("="*70)
    
    results = []
    errors = 0
    
    for idx, row in df_atg_stop.iterrows():
        if idx % 10000 == 0:
            print(f"  Processed {idx:,} / {len(df_atg_stop):,} regions...")
        
        score = score_region_with_riboseq(row, transcript_data, fw_bw, rv_bw)
        results.append(score)
        
        if score.get('error'):
            errors += 1
    
    # Add scores to dataframe
    df_atg_stop['total_riboseq_counts'] = [r['total_counts'] for r in results]
    df_atg_stop['frame0_riboseq_counts'] = [r['frame0_counts'] for r in results]
    df_atg_stop['frame1_riboseq_counts'] = [r['frame1_counts'] for r in results]
    df_atg_stop['frame2_riboseq_counts'] = [r['frame2_counts'] for r in results]
    df_atg_stop['riboseq_density'] = [r['riboseq_density'] for r in results]
    
    print(f"\n✓ Scored {len(df_atg_stop):,} regions")
    print(f"  Errors: {errors:,} regions")
    print(f"  Regions with counts > 0: {(df_atg_stop['total_riboseq_counts'] > 0).sum():,}")
else:
    print("⚠️  Skipping scoring - bigWig files not available")

## 9. Calculate Frame Balance Metrics

In [ ]:
def calculate_frame_balance(row):
    """Calculate frame balance metric: how evenly distributed are counts across frames?
    
    Returns value between 0 (all in one frame) and 1 (perfectly balanced)
    """
    total = row['total_riboseq_counts']
    if total == 0:
        return 0.0
    
    f0 = row['frame0_riboseq_counts'] / total
    f1 = row['frame1_riboseq_counts'] / total
    f2 = row['frame2_riboseq_counts'] / total
    
    # Use entropy-based measure: H = -sum(p * log(p))
    # Normalize to 0-1 range (max entropy for 3 categories is log(3))
    entropy = 0.0
    for p in [f0, f1, f2]:
        if p > 0:
            entropy -= p * np.log(p)
    
    max_entropy = np.log(3)  # Maximum entropy for 3 categories
    balance = entropy / max_entropy if max_entropy > 0 else 0.0
    
    return balance

if fw_bw and rv_bw:
    df_stop_free['frame_balance'] = df_stop_free.apply(calculate_frame_balance, axis=1)
    df_atg_stop['frame_balance'] = df_atg_stop.apply(calculate_frame_balance, axis=1)
    
    print("Frame balance metrics calculated")
    print("  (0 = all counts in one frame, 1 = perfectly balanced across 3 frames)")

In [ ]:
if fw_bw and rv_bw:
    print("\n" + "="*70)
    print("SRD5A1 RIBO-SEQ ANALYSIS")
    print("="*70)
    
    for analysis_name, df in [('Stop-Free', df_stop_free), ('ATG-to-STOP', df_atg_stop)]:
        print(f"\n{analysis_name}:")
        srd5a1 = df[df['gene_name'] == 'SRD5A1'].copy()
        
        if len(srd5a1) > 0:
            # Sort by total counts
            srd5a1 = srd5a1.sort_values('total_riboseq_counts', ascending=False)
            
            print(f"  ✅ Found {len(srd5a1)} regions")
            
            best = srd5a1.iloc[0]
            print(f"\n  Best region (by Ribo-Seq coverage):")
            print(f"    Transcript: {best['transcript_id'][:50]}...")
            print(f"    Length: {best['region_length_codons']:.0f} codons")
            print(f"    CDS context: {best['cds_context']}")
            print(f"    Total Ribo-Seq counts: {best['total_riboseq_counts']:.1f}")
            print(f"    Ribo-Seq density: {best['riboseq_density']:.2f} counts/codon")
            print(f"\n    Per-frame counts:")
            print(f"      Frame 0: {best['frame0_riboseq_counts']:.1f}")
            print(f"      Frame 1: {best['frame1_riboseq_counts']:.1f}")
            print(f"      Frame 2: {best['frame2_riboseq_counts']:.1f}")
            print(f"    Frame balance: {best['frame_balance']:.3f}")
        else:
            print("  ❌ Not found")

In [ ]:
if fw_bw and rv_bw:
    # Save both analyses with Ribo-Seq scores
    output_dir = Path('../results/triple_frame_riboseq')
    output_dir.mkdir(parents=True, exist_ok=True)
    
    df_stop_free.to_csv(output_dir / 'stop_free_with_riboseq.tsv', sep='\t', index=False)
    print(f"\nSaved: {output_dir / 'stop_free_with_riboseq.tsv'}")
    
    df_atg_stop.to_csv(output_dir / 'atg_stop_with_riboseq.tsv', sep='\t', index=False)
    print(f"Saved: {output_dir / 'atg_stop_with_riboseq.tsv'}")
    
    print("\n✓ All results saved with Ribo-Seq scores!")
else:
    print("⚠️  Skipping save - scoring not completed")

In [ ]:
if fw_bw and rv_bw:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    datasets = [
        (df_stop_free, 'Stop-Free'),
        (df_atg_stop, 'ATG→STOP')
    ]
    
    # Top row: Ribo-Seq density distribution
    for idx, (df, title) in enumerate(datasets):
        ax = axes[0, idx]
        
        # Filter for regions with counts > 0
        df_with_counts = df[df['total_riboseq_counts'] > 0]
        
        if len(df_with_counts) > 0:
            densities = df_with_counts['riboseq_density']
            ax.hist(np.log10(densities + 0.01), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            ax.set_xlabel('log10(Ribo-Seq density)', fontsize=10)
            ax.set_ylabel('Frequency', fontsize=10)
            ax.set_title(f'{title}\n({len(df_with_counts):,} regions with coverage)', fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, 'No regions\nwith coverage', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title, fontsize=11, fontweight='bold')
    
    # Bottom row: Frame balance distribution
    for idx, (df, title) in enumerate(datasets):
        ax = axes[1, idx]
        
        df_with_counts = df[df['total_riboseq_counts'] > 0]
        
        if len(df_with_counts) > 0:
            balance = df_with_counts['frame_balance']
            ax.hist(balance, bins=30, color='darkorange', edgecolor='black', alpha=0.7)
            ax.set_xlabel('Frame Balance', fontsize=10)
            ax.set_ylabel('Frequency', fontsize=10)
            ax.set_title(f'Frame Balance Distribution\n(0=unbalanced, 1=balanced)', fontsize=10)
            ax.grid(True, alpha=0.3)
            ax.axvline(balance.median(), color='red', linestyle='--', linewidth=2, label=f'Median={balance.median():.2f}')
            ax.legend(fontsize=8)
        else:
            ax.text(0.5, 0.5, 'No regions\nwith coverage', ha='center', va='center', transform=ax.transAxes)
    
    plt.tight_layout()
    
    fig_path = '../results/figures/riboseq_analysis.png'
    Path(fig_path).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.savefig(fig_path.replace('.png', '.pdf'), bbox_inches='tight')
    print(f"\nSaved figure: {fig_path}")
    plt.show()
else:
    print("⚠️  Skipping figure generation - scoring not completed")

In [ ]:
if fw_bw:
    fw_bw.close()
    print("Closed forward bigWig")

if rv_bw:
    rv_bw.close()
    print("Closed reverse bigWig")